In [4]:
import pandas as pd
from pathlib import Path
import re

In [5]:
# Define paths
csv_input = Path(r"C:\Archive\development\2026_thecall\code\metadataanalysis\combined_metadata.csv")
pdf_folder = Path(r"C:\Archive\development\2026_thecall\2026_thecall\downloads")
output_filtered_csv = Path(r"C:\Archive\development\2026_thecall\code\metadataanalysis\matched_metadata.csv")

# Define our paths
source_dir = Path(r"C:\Archive\development\2026_thecall\2026_thecall\metadata1")
output_dir = Path(r"C:\Archive\development\2026_thecall\code\metadataanalysis")
output_file = output_dir / "combined_metadata.csv"


In [6]:
def combine_excel_files():
    # Ensure the output directory exists
    output_dir.mkdir(parents=True, exist_ok=True)
    
    # Use rglob to find all Excel files in subdirectories
    # This captures both .xlsx and .xls
    excel_files = list(source_dir.rglob("*.xls*"))
    
    if not excel_files:
        print("No Excel files found in the specified directory.")
        return

    print(f"Found {len(excel_files)} files. Starting assembly...")

    # List to hold dataframes
    all_data = []

    for file in excel_files:
        try:
            # Read the excel file
            df = pd.read_excel(file)
            # Optional: Add a column to track which file the data came from
            df['source_file'] = file.name
            all_data.append(df)
        except Exception as e:
            print(f"Could not read {file.name}: {e}")

    # Combine everything
    if all_data:
        combined_df = pd.concat(all_data, ignore_index=True)
        
        # Save to CSV
        combined_df.to_csv(output_file, index=False)
        print(f"Success! Combined file saved to: {output_file}")

if __name__ == "__main__":
    combine_excel_files()

Found 8 files. Starting assembly...
Success! Combined file saved to: C:\Archive\development\2026_thecall\code\metadataanalysis\combined_metadata.csv


In [7]:
def match_pdfs_to_csv():
    # 1. Load the existing CSV
    if not csv_input.exists():
        print(f"Error: {csv_input} not found. Run the previous script first.")
        return
    
    df = pd.read_csv(csv_input)
    # Ensure StoreId is treated as a string to avoid matching 101 with 101.0
    df['StoreId'] = df['StoreId'].astype(str).str.strip()

    # 2. Map the PDF files in the folder
    # We create a set of StoreIds found in the filenames for O(1) lookup speed
    pdf_files = list(pdf_folder.glob("*.pdf"))
    
    # Store found IDs and a map of ID -> FileName for reporting
    found_ids = {}
    for f in pdf_files:
        # Extract only the digits (e.g., "123.pdf" -> "123")
        store_id_match = re.search(r'(\d+)', f.stem)
        if store_id_match:
            found_ids[store_id_match.group(1)] = f.name

    # 3. Filter the DataFrame
    # Keep rows where the StoreId exists in our found_ids dictionary
    matched_df = df[df['StoreId'].isin(found_ids.keys())].copy()
    
    # 4. Identify unmatched files
    csv_store_ids = set(df['StoreId'].unique())
    unmatched_files = [fname for sid, fname in found_ids.items() if sid not in csv_store_ids]

    # 5. Output Results
    matched_df.to_csv(output_filtered_csv, index=False)
    
    print(f"Filtering Complete.")
    print(f"Rows in original CSV: {len(df)}")
    print(f"Rows with matching PDFs: {len(matched_df)}")
    print(f"Filtered CSV saved to: {output_filtered_csv}")

    if unmatched_files:
        print("\n--- Files with no match in CSV ---")
        for f in unmatched_files:
            print(f"- {f}")
    else:
        print("\nAll PDF files successfully matched a row in the CSV.")

if __name__ == "__main__":
    match_pdfs_to_csv()

Filtering Complete.
Rows in original CSV: 730
Rows with matching PDFs: 337
Filtered CSV saved to: C:\Archive\development\2026_thecall\code\metadataanalysis\matched_metadata.csv

All PDF files successfully matched a row in the CSV.
